<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_71_triton_inference_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🚦 **Module 71 — Triton Inference Server** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 9 · Production Gaps


# Module 71 — Triton Inference Server

> **vLLM** (M44) is the right answer for *one model, one type — LLMs on NVIDIA*. **TF Serving** (M40) is the right answer for *one model, one type — Keras*. What if you have ten models — an embedder + a reranker + a tabular XGBoost + an OCR detector + a vision-language model + an LLM — and want **one server, one URL, one set of metrics**? That's **Triton Inference Server**: NVIDIA's open-source, multi-framework, hardware-aware inference engine. It's the layer underneath SageMaker NeoTriton, Vertex AI Prediction, and most NVIDIA NIM endpoints.

### What you'll cover
1. The problem Triton solves
2. The **model repository** — file-system convention is the API
3. **`config.pbtxt`** — the one config file you'll write a lot
4. Multi-framework backends — **PyTorch · TensorFlow · ONNX · TensorRT · vLLM · Python**
5. **Dynamic batching** — Triton's signature throughput win
6. **Multi-instance** + GPU isolation
7. **Ensemble** + **Business Logic Scripting (BLS)** — pipelines inside the server
8. **TensorRT** and **TensorRT-LLM** — the absolute-fastest backend
9. Metrics, **Model Analyzer**, and shipping decisions
10. Triton vs vLLM vs TF Serving vs TorchServe — when to pick what


## 1 · The problem

A real ML platform has dozens of models in production:

```
   /v1/embed          → sentence-transformers (PyTorch)
   /v1/rerank         → cross-encoder (PyTorch)
   /v1/classify       → fraud XGBoost (sklearn / ONNX)
   /v1/ocr/detect     → YOLO (ONNX / TensorRT)
   /v1/ocr/recognise  → TrOCR (PyTorch)
   /v1/caption        → Qwen2-VL (vLLM)
   /v1/chat           → Llama-3 (vLLM / TRT-LLM)
   /v1/asr            → Whisper (PyTorch / TensorRT)
   /v1/tts            → SpeechT5 (PyTorch / ONNX)
```

You **could** run nine servers behind nine k8s Deployments. Most teams do at first. Then the problems pile up:
- **Nine container images** to maintain — different CUDA versions, different runtimes.
- **Nine sets of metrics** to wire into Prometheus.
- **Nine pods** to scale; nine cold-start configurations.
- **Cross-model latency** — embedder + LLM in one call = two HTTP round-trips.

**Triton solves this**: **one server**, **one Docker image**, **many models**, **one HTTP/gRPC API**, **one Prometheus exporter**. It hot-loads / unloads models from a model repository on the fly.

## 2 · The model repository — file system is the API

Triton has no admin GUI. You drop files into a directory; Triton serves them.

```
models/
├── classify/                            ← model name = URL slug
│   ├── config.pbtxt
│   └── 1/                                ← version directory
│       └── model.onnx
├── embed/
│   ├── config.pbtxt
│   └── 1/
│       └── model.pt                     ← TorchScript
├── llama3/
│   ├── config.pbtxt
│   ├── 1/
│   └── 2/                               ← live alongside v1, hot-pinnable
│       └── model.plan                   ← TensorRT engine
└── pipeline/
    ├── config.pbtxt                      ← ensemble: glue several models together
    └── 1/
```

A few rules that are easy to miss:
- The directory name is the **model name** (used in the URL).
- The numbered subdirs are **versions**. Triton picks the highest numeric by default; can pin via config.
- `config.pbtxt` is **protobuf text** — quirky syntax but stable.
- Drop a new version dir → Triton auto-loads it. Remove one → Triton auto-unloads it. **No restart.**

## 3 · `config.pbtxt` — the only config you'll write a lot

In [ ]:
sample_config = '''
# models/embed/config.pbtxt
name: "embed"
backend: "pytorch"
max_batch_size: 64

input [
  {
    name: "INPUT__0"
    data_type: TYPE_INT64
    dims: [ -1 ]                     # variable sequence length
  }
]

output [
  {
    name: "OUTPUT__0"
    data_type: TYPE_FP32
    dims: [ 384 ]                    # MiniLM embedding dim
  }
]

# THE single most important block — see §5
dynamic_batching {
  preferred_batch_size: [ 8, 16, 32 ]
  max_queue_delay_microseconds: 5000
}

# How many copies of this model on each GPU — see §6
instance_group [
  {
    count: 2
    kind: KIND_GPU
  }
]

# Optional: pin to specific versions
version_policy { specific { versions: [1] } }
'''
print(sample_config)

**Five `config.pbtxt` knobs you'll touch every time.**

1. **`backend`** — `pytorch`, `tensorflow`, `onnxruntime`, `tensorrt`, `vllm`, `python`, `dali`.
2. **`max_batch_size`** — > 0 enables batching; 0 = no batching.
3. **`input` / `output`** — names match what the model exports; **`-1`** means dynamic dim.
4. **`dynamic_batching`** — turn on; tune `preferred_batch_size` + `max_queue_delay_microseconds`.
5. **`instance_group`** — how many copies of the model per device.

## 4 · Multi-framework backends

In [ ]:
backends_table = '''
# Snippet of valid `backend:` values + what they expect under v1/

backend: "pytorch"        →  v1/model.pt          (TorchScript) — or  model.py for stateless PyTorch via python backend
backend: "tensorflow"     →  v1/model.savedmodel/ (SavedModel)
backend: "onnxruntime"    →  v1/model.onnx        (ONNX)
backend: "tensorrt"       →  v1/model.plan        (TRT engine — see §8)
backend: "openvino"        →  v1/model.xml        (Intel CPU/iGPU)
backend: "fil"             →  v1/xgboost.model    (Forest Inference Library — XGBoost / LightGBM / cuML)
backend: "vllm"            →  v1/model.json       (vLLM engine — see §10)
backend: "python"          →  v1/model.py         (arbitrary Python class — escape hatch)
backend: "dali"            →  v1/dali.json        (NVIDIA DALI — preprocessing pipelines on GPU)
backend: "tensorrt_llm"    →  v1/<engine dir>      (TRT-LLM — see §8)
'''
print(backends_table)

**`backend: "python"`** is the universal escape hatch — wrap arbitrary Python in a `TritonPythonModel` class:

```python
class TritonPythonModel:
    def initialize(self, args): ...           # load weights once
    def execute(self, requests):              # called per batch
        # parse, run model, return responses
        return responses
```

Useful for: orchestration logic, custom preprocessing, calling an external API as a model.

**For LLMs specifically:** **`vllm`** backend wraps the M44 engine — you get **PagedAttention** + **continuous batching** **inside Triton**, with all of Triton's metrics + routing on top.

## 5 · Dynamic batching — Triton's signature win

The single biggest reason teams pick Triton over a hand-rolled FastAPI server.

```
   Without dynamic batching:
     req A → model.forward([A])  →  reply A
     req B → model.forward([B])  →  reply B
     req C → model.forward([C])  →  reply C   ← three GPU calls

   With dynamic batching (max_queue_delay=5ms):
     req A ─┐
     req B ─┼─►  model.forward([A, B, C])  →  reply A, B, C   ← ONE GPU call
     req C ─┘
```

Triton holds requests in a queue for up to `max_queue_delay_microseconds`, then flushes whatever has arrived as a single batch up to `preferred_batch_size`. **2-10× throughput on the same hardware**, configurable per model.

### Three tuning rules
- **Latency-critical (chat, search)**: `max_queue_delay_microseconds = 1000-5000` (1-5 ms tolerable).
- **Throughput-critical (batch embedding)**: `max_queue_delay_microseconds = 20000+`, large `preferred_batch_size`.
- **List `preferred_batch_size` in ascending order** — Triton picks the largest one ≤ the queued count.

> 🟡 Dynamic batching requires **stateless** inputs (no token-by-token decoding). For LLMs, use the **vllm** backend (continuous batching) — same effect, transformer-specific.

## 6 · Multi-instance + GPU isolation

In [ ]:
multi_instance_examples = '''
# Two copies on every GPU — uses tensor cores in parallel for small models
instance_group [
  { count: 2, kind: KIND_GPU }
]

# Pin to specific GPUs (e.g. dedicate GPU 0 to LLM, GPU 1 to embedder)
instance_group [
  { count: 1, kind: KIND_GPU, gpus: [1] }
]

# Mix CPU + GPU copies (e.g. fallback when GPUs are saturated)
instance_group [
  { count: 2, kind: KIND_GPU },
  { count: 2, kind: KIND_CPU }
]

# Multiple model groups — same model, different rate-budget tiers
instance_group [
  { name: "high_prio", count: 1, kind: KIND_GPU, gpus: [0] },
  { name: "low_prio",  count: 1, kind: KIND_GPU, gpus: [1] }
]
'''
print(multi_instance_examples)

**When to bump `count > 1`.** If GPU utilisation stays low under load (< 60 %) but latency rises, the model is bottlenecked by **launch + memory copy overhead**, not by compute. A second instance running concurrently uses the otherwise-idle compute. Empirically, 2-4 instances per GPU is the sweet spot for small models (< 1 B params); 1 instance per GPU for 7B+ LLMs.

## 7 · Ensembles + Business Logic Scripting (BLS)

A production AI request rarely calls *one* model. RAG = embed → search → rerank → LLM. OCR = detect → recognise → spellcheck. Triton has two ways to chain models **inside the server**, without an extra HTTP hop:

### Ensemble — DAG defined in `config.pbtxt`
Simple linear or DAG pipelines. Zero Python.

```
   request ──► [preproc] ──► [embed] ──► [rerank] ──► response
```

```
# models/rag_pipeline/config.pbtxt — abbreviated
name: "rag_pipeline"
platform: "ensemble"
input [{ name: "QUERY", data_type: TYPE_STRING, dims: [1] }]
output [{ name: "TOP_K", data_type: TYPE_INT32, dims: [-1] }]
ensemble_scheduling {
  step [
    {
      model_name: "embed"
      model_version: -1
      input_map  { key: "INPUT__0" value: "QUERY" }
      output_map { key: "OUTPUT__0" value: "QUERY_VEC" }
    },
    {
      model_name: "ann_search"
      model_version: -1
      input_map  { key: "INPUT__0" value: "QUERY_VEC" }
      output_map { key: "OUTPUT__0" value: "TOP_K" }
    }
  ]
}
```

### BLS — Python-backend model that calls other models
For branching, loops, conditionals, anything ensembles can't express.

```python
import triton_python_backend_utils as pb_utils

class TritonPythonModel:
    def execute(self, requests):
        responses = []
        for r in requests:
            q = pb_utils.get_input_tensor_by_name(r, "QUERY")
            emb = pb_utils.InferenceRequest("embed", ["OUTPUT__0"], [q]).exec()
            top = pb_utils.InferenceRequest("ann_search", ["OUTPUT__0"], [emb]).exec()
            # branch: if confidence low, escalate to LLM-rerank
            if top.confidence() < 0.3:
                top = pb_utils.InferenceRequest("llm_rerank", ...).exec()
            responses.append(pb_utils.InferenceResponse(...))
        return responses
```

**Why both matter.** Keep the orchestration **inside Triton** and you save network hops + serialise once instead of N times. A four-model RAG pipeline often runs **3× faster** as a Triton ensemble than as four HTTP services.

## 8 · TensorRT and TensorRT-LLM — the absolute-fastest backend

Triton can serve a PyTorch model directly. But for production NVIDIA hardware, compiling to **TensorRT** is the difference between "good" and "as fast as physics allows."

### TensorRT (general)
- Ahead-of-time compiler — fuses kernels, picks tactics, runs in INT8 / FP8 / FP16.
- Input: ONNX or PyTorch (`torch_tensorrt`); output: `.plan` engine file.
- **2-10× faster** than PyTorch eager on the same GPU.
- Tied to a **specific GPU architecture** — re-compile per arch (e.g. one engine for A100, another for H100).

### TensorRT-LLM (LLM-specific)
- TRT optimised for transformer inference.
- **In-flight batching** (Triton's name for continuous batching), **paged KV cache**, **speculative decoding**, **quantisation** (FP8, INT4 AWQ).
- Backend in Triton: `tensorrt_llm`.
- For NVIDIA-only deployments, TRT-LLM + Triton is **measurably faster than vLLM** at the cost of a hairy build process. (vLLM is more portable + easier.)

**Practical pattern:** **vLLM** until you have proof TRT-LLM's extra 20-30% throughput matters. **TRT-LLM** when every token/sec is money.

## 9 · Metrics + Model Analyzer

### Metrics
Triton exposes Prometheus metrics at `/metrics` out of the box (M50):

```
nv_inference_request_success
nv_inference_request_failure
nv_inference_count                  # total requests per model
nv_inference_exec_count             # number of GPU exec calls
nv_inference_request_duration_us    # request latency histogram
nv_inference_queue_duration_us      # time spent in dynamic batch queue
nv_inference_compute_input_duration_us
nv_inference_compute_infer_duration_us
nv_inference_compute_output_duration_us
nv_gpu_utilization
nv_gpu_memory_used_bytes
```

The breakdown is the gold here: latency = queue + compute_input + compute_infer + compute_output. If `queue_duration` is high, dynamic batching is undersized. If `compute_infer` is high, the model itself is the bottleneck.

### Model Analyzer
Triton ships **`model-analyzer`** — a CLI that **automatically sweeps** `instance_group`, `max_batch_size`, `dynamic_batching` settings for one model and produces a CSV + chart of throughput-vs-latency Pareto curves.

```bash
$ model-analyzer profile \
    --model-repository /models \
    --model-names embed,classify,llama3 \
    --triton-launch-mode docker \
    --override-output-model-repo-path /tmp/output_models
```

A morning's run can save weeks of hand-tuning.

## 10 · Triton vs vLLM vs TF Serving vs TorchServe

| Server | Best at | Watch out for |
|---|---|---|
| **Triton** | multi-model, multi-framework, NVIDIA, complex pipelines | YAML-shaped pbtxt; learning curve; NVIDIA-centric |
| **vLLM** (M44) | LLMs only, fast iteration, PagedAttention | one model per process; less polished metrics |
| **TGI** (HuggingFace) | LLMs, HF integration, simple HTTP | HF licence terms on newer versions |
| **TF Serving** (M40) | TensorFlow / Keras only | TF-only; aging UX |
| **TorchServe** | PyTorch only | maintenance-mode; declining mindshare |
| **Ray Serve** | mixed Python apps; streaming pipelines | heavier per-pod footprint |
| **SageMaker / Vertex / NIM** | hosted Triton with autoscaling | vendor lock-in; cost; sometimes lagging features |

### When to pick Triton
- ✅ NVIDIA fleet; many heterogeneous models behind one API.
- ✅ Want one set of Prometheus metrics + one IAM surface.
- ✅ Need **ensembles** or **BLS** to fuse pipelines into a single call.
- ✅ Want **TensorRT** / **TRT-LLM** speed without writing a server.

### When NOT to pick Triton
- ❌ One model, one type — vLLM / TGI is simpler.
- ❌ AMD or Apple Silicon — Triton support is limited; use **vLLM** + ROCm or **MLX**.
- ❌ "Just put a model behind FastAPI" — Triton's overhead isn't justified.

### Anti-patterns
- ❌ Wrapping every model with `backend: python` — kills the throughput advantage.
- ❌ One config_pbtxt with `max_batch_size: 0` everywhere — you disabled batching.
- ❌ Using **ensembles** for branching logic — they only do DAGs; use BLS for `if/else`.
- ❌ Letting one big LLM and a small embedder share a GPU without `instance_group` pinning — long LLM call starves the embedder.

## ✅ Recap
- Triton = **one server for many models** across PyTorch / TF / ONNX / TensorRT / vLLM / Python.
- **Model repository** is the API: directories, `config.pbtxt`, hot-load/unload.
- **Dynamic batching** is the signature throughput win — 2-10× on the same hardware.
- **Instance groups** = per-GPU model parallelism; bump to 2-4 for small models.
- **Ensembles** + **BLS** keep multi-model pipelines inside the server (no HTTP hops).
- **TensorRT / TRT-LLM** = absolute fastest, NVIDIA-only, harder to build.
- Pick Triton when you have **many heterogeneous models** on NVIDIA; pick vLLM/TGI when it's one LLM; pick TF Serving / TorchServe in single-framework shops.

Next: **M72 — Computer-use / Browser Agents**.
